# 03_preprocess — spaCy sentence segmentation + NER

> **⚑ Islamic Republic corpus (IRNA / Tasnim).** Isolated copy of the English notebook — reads `data/iran_raw/`, writes `data/interim/iran/` + `data/output/iran/`, and (where actor normalization is involved) uses `src/alias_map_iran`. The English pipeline and its outputs are untouched.


> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/interim/corpus_clean.jsonl`  
**Output:** `data/interim/sentences.jsonl`

This is the most computationally expensive step. Run it once; downstream notebooks read its output without re-running spaCy.

## Pipeline steps in this notebook

1. Setup & paths
2. Load spaCy transformer model (`en_core_web_trf`)
3. Load cleaned corpus
4. Run spaCy pipeline (sentence segmentation + NER + lemmas)
5. Quality report
6. Spot-check sample sentences
7. Write sentences.jsonl

## Step 1: Setup & paths

In [ ]:
import json
import sys
import time
from pathlib import Path
from collections import Counter

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim' / 'iran'
IN_FILE     = INTERIM_DIR / 'corpus_clean.jsonl'
OUT_FILE    = INTERIM_DIR / 'sentences.jsonl'

print(f'Python : {sys.executable}')
print(f'Input  : {IN_FILE}')
print(f'Output : {OUT_FILE}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'
assert IN_FILE.exists(),        f'ERROR: {IN_FILE} not found — run 02_clean first'

## Step 2: Load spaCy transformer model

Uses `en_core_web_trf` — the transformer-based pipeline. 
Best NER quality on news text (Balluff, Boomgaarden & Waldherr 2024). 
The lemmatizer is **enabled**: notebook 04 matches single-word concept entries against token lemmas, so every sentence record stores a `lemmas` list alongside the raw `text`.

In [ ]:
import spacy

nlp = spacy.load('en_core_web_trf')
print('Loaded en_core_web_trf')
print(f'Pipeline components: {nlp.pipe_names}')

## Step 3: Load cleaned corpus

In [ ]:
with open(IN_FILE, encoding='utf-8') as f:
    articles = [json.loads(line) for line in f]

print(f'Loaded {len(articles)} articles')
required = {'id', 'date', 'source', 'window', 'body'}
missing  = [a['id'] for a in articles if not required.issubset(a.keys())]
assert not missing, f'{len(missing)} articles missing required fields: {missing[:5]}'
print('All articles have required fields.')

## Step 4: Run spaCy pipeline

`nlp.pipe(batch_size=32)` is well-tuned for ~2k articles on a standard laptop. 
Progress and ETA print every 100 articles.

In [ ]:
KEEP_LABELS = {'GPE', 'ORG', 'PERSON', 'NORP'}
BATCH_SIZE  = 32

sentences = []
entity_counter = Counter()
t_start = time.time()

bodies = [a['body'] for a in articles]

for idx, (doc, art) in enumerate(
    zip(nlp.pipe(bodies, batch_size=BATCH_SIZE), articles)
):
    if (idx + 1) % 100 == 0 or idx == 0:
        elapsed = time.time() - t_start
        rate    = (idx + 1) / max(elapsed, 1)
        eta     = (len(articles) - idx - 1) / max(rate, 1)
        print(f'  Article {idx+1:4d}/{len(articles)}  '
              f'elapsed={elapsed:.0f}s  rate={rate:.1f} art/s  ETA={eta:.0f}s')

    for i, sent in enumerate(doc.sents):
        text = sent.text.strip()
        if not text:
            continue
        ents = [
            [e.text, e.label_]
            for e in sent.ents
            if e.label_ in KEEP_LABELS
        ]
        for _, label in ents:
            entity_counter[label] += 1

        sentences.append({
            'sentence_id': f"{art['id']}_{i:03d}",
            'article_id':  art['id'],
            'date':        art['date'],
            'source':      art['source'],
            'window':      art['window'],
            'text':        text,
            'lemmas':      [t.lemma_.lower() for t in sent if not t.is_space],
            'ents':        ents,
        })

total_time = time.time() - t_start
print(f'\nDone. {len(sentences)} sentences from {len(articles)} articles in {total_time:.1f}s')

## Step 5: Quality report

In [ ]:
avg_sents  = len(sentences) / max(len(articles), 1)
ids        = [s['sentence_id'] for s in sentences]
unique_ids = len(set(ids))

print(f'Total sentences          : {len(sentences)}')
print(f'Avg sentences/article    : {avg_sents:.1f}')
print(f'Unique sentence IDs      : {unique_ids}')
print(f'Duplicate IDs            : {len(ids) - unique_ids}  (must be 0)')
print()
print('Entity label counts (KEEP_LABELS only):')
for label, cnt in entity_counter.most_common():
    print(f'  {label:8s}  {cnt}')

if len(ids) != unique_ids:
    print('WARNING: Duplicate sentence_ids — investigate ID reassignment in 02_clean.')
if avg_sents < 10:
    print('WARNING: Very low avg sentences/article. Body may be truncated.')
elif avg_sents > 60:
    print('WARNING: Very high avg sentences/article. Boilerplate may not be fully stripped.')

## Step 6: Spot-check sample sentences

In [ ]:
import random
has_ents = [s for s in sentences if s['ents']]
sample = random.sample(has_ents, min(5, len(has_ents)))
print('--- Sample sentences with NER output ---')
for s in sample:
    print(f"\n  sentence_id : {s['sentence_id']}")
    print(f"  window      : {s['window']}")
    print(f"  text        : {s['text'][:200]}")
    print(f"  lemmas      : {s['lemmas'][:18]}")
    print(f"  ents        : {s['ents']}")

## Step 7: Write sentences.jsonl

In [ ]:
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for sent in sentences:
        f.write(json.dumps(sent, ensure_ascii=False) + '\n')

print(f'Wrote {len(sentences)} sentences to {OUT_FILE}')
print()
print('VALIDATION CHECKPOINT (03_preprocess):')
print(f'  Avg sentences/article : {avg_sents:.1f}')
print(f'  Unique sentence IDs   : {unique_ids == len(ids)}  -> must be True')
print()
print('NOTE: Do not re-run this notebook unless you change the corpus.')
print('      Notebooks 04–07 can re-run freely without re-running spaCy.')